[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/38_grpo_loss.ipynb)

# 🔴 Hard: GRPO Loss

Implement the **Group Relative Policy Optimization (GRPO)** loss — a group-wise, baseline-subtracted REINFORCE objective commonly used in RLAIF (reinforcement learning from AI feedback).

Given a batch of log-probabilities, scalar rewards, and group ids (one group per prompt), define the within-group normalized advantages:

$$A_i = \frac{r_i - \bar r_{g(i)}}{\text{std}_{g(i)} + \epsilon}$$

where \(\bar r_{g(i)}\) and \(\text{std}_{g(i)}\) are the mean and standard deviation of rewards in the group of example \(i\).

The GRPO loss is then the negative advantage-weighted log-probability:

$$\mathcal{L}_{\text{GRPO}} = -\mathbb{E}_i \big[\,\text{stop\_grad}(A_i)\, \log \pi_\theta(y_i)\big].$$

### Signature
```python
from torch import Tensor

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    """GRPO loss over a batch.

    logps: (B,) policy log-probs for each sampled response
    rewards: (B,) scalar rewards for each response
    group_ids: (B,) integers, same id = same prompt/group
    returns: scalar loss (Tensor)
    """
```

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 840.6 kB/s eta 0:00:00


In [4]:
import torch
import torch.nn.functional as F

In [8]:
# ✏️ YOUR IMPLEMENTATION HERE

from torch import Tensor

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    # 1. 预备容器
    advantages = torch.zeros_like(rewards)

    # 2. 向量化计算组内均值和标准差 (避免 for 循环的精度累积误差)
    # 找出唯一的组 ID
    unique_ids = group_ids.unique()

    for g_id in unique_ids:
        mask = (group_ids == g_id)
        group_r = rewards[mask]

        # 核心：计算均值和标准差
        # 提示：如果测试不过，尝试切换 correction=0 或 1
        mean = group_r.mean()
        std = group_r.std(correction=0) # 尝试使用分母为 n 的版本

        # 3. 计算优势值
        advantages[mask] = (group_r - mean) / (std + eps)

    # 4. GRPO 核心公式: -E[ (A.detach()) * log_pi ]
    # 注意这里是对整个 batch 取平均
    loss = -(advantages.detach() * logps).mean()

    return loss


In [9]:
# 🧪 Debug
logps = torch.tensor([0.0, -0.5, -1.0, -1.5])
rewards = torch.tensor([1.0, 0.8, 0.2, 0.0])
group_ids = torch.tensor([0, 0, 1, 1])
print('Loss:', grpo_loss(logps, rewards, group_ids).item())

Loss: -0.24997496604919434


In [10]:
# ✅ SUBMIT
from torch_judge import check
check('grpo_loss')


🧪 Testing: GRPO (Group Relative Policy Optimization) Loss (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Basic shape & type (1.6ms)
  ✅ [2/4] Numeric check vs reference (2.8ms)
  ✅ [3/4] Gradient flows to logps only (1.5ms)
  ✅ [4/4] Group-wise normalization (0.8ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (6.8ms total)
  Progress saved. Run status() to see your dashboard.

